# Google Trends Search Pipeline — Step by Step

Extractor CSV (`date`, `location`, `topic`, `value`) → EW-keyed search index
ready to join SINAN and climate, one step at a time.

Mirrors `src/pipeline/google_trends/` exactly — this notebook unrolls `build()`
so each transformation is visible. Runs on the committed fixture in
`tests/fixtures/google_trends/` (tiny sample), so it executes instantly.

**Pipeline:** scan → rename keys → parse `ew` → floor to Sunday → optional drop BR → order

**Literature / project alignment:**

| Choice | Grounding |
|---|---|
| Weekly grain already Sunday-dated | Borges et al. *Sci Data* data paper; GT extractor window |
| Join keys `ew` + `state_abbrev` | Same as SINAN / climate transforms |
| Shared `floor_to_sunday` | One epi-week rule across sources |
| Relative index 0–100, no re-normalisation | Google Trends semantics; cross-UF values are not absolute volume |
| Related topics/queries deferred | Monthly; extraction incomplete — separate stage |
| Thin transform (no topic filter / imputation) | Collection owns vocabulary (`popular_terms.csv` + Freebase IDs) |


## Setup

In [1]:
import sys
from pathlib import Path

import polars as pl

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from pipeline.google_trends.spec import OUTPUT_COLUMNS, STATE_ABBREV, EW, TOPIC, VALUE
from pipeline.google_trends.steps import (
    build,
    drop_national,
    floor_ew,
    order_columns,
    parse_ew,
    rename_keys,
    scan_search,
)

pl.Config.set_tbl_rows(12)

FIXTURES = PROJECT_ROOT / "tests" / "fixtures" / "google_trends"
FU_PATH = PROJECT_ROOT / "data" / "epidemiological" / "br_federative_units.csv"
SAMPLE = FIXTURES / "GoogleTrends_search_sample.csv"
SAMPLE.exists()


True

## Literature checklist (what this notebook must prove)

1. Extractor weeks are already **Sundays** (or become Sundays after `floor_to_sunday`).
2. Keys rename to **`ew` / `state_abbrev`** — identical join axis as climate and SINAN.
3. Values stay in **[0, 100]** (or null); no cross-UF rescaling in this stage.
4. National `BR` rows are optional (kept by default for national curves).
5. Related topics/queries are **out of scope** for v1 (monthly grain).


## Step 1 — Scan the extractor CSV

`scan_search` opens a lazy CSV with string dates and float values. The real
file is `data/google_trends/GoogleTrends_search.csv` (~6.5 MB, 28 geos × ~30 topics).


In [2]:
raw = scan_search(SAMPLE)
raw.collect_schema(), raw.collect()


(Schema([('date', String),
         ('location', String),
         ('topic', String),
         ('value', Float64)]),
 shape: (4, 4)
 ┌────────────┬──────────┬────────┬───────┐
 │ date       ┆ location ┆ topic  ┆ value │
 │ ---        ┆ ---      ┆ ---    ┆ ---   │
 │ str        ┆ str      ┆ str    ┆ f64   │
 ╞════════════╪══════════╪════════╪═══════╡
 │ 2020-01-26 ┆ AC       ┆ dengue ┆ 12.0  │
 │ 2020-01-26 ┆ SP       ┆ dengue ┆ 45.0  │
 │ 2020-01-26 ┆ BR       ┆ dengue ┆ 30.0  │
 │ 2020-02-02 ┆ AC       ┆ dengue ┆ 15.0  │
 └────────────┴──────────┴────────┴───────┘)

## Step 2 — Rename keys (`date` → `ew`, `location` → `state_abbrev`)

Aligns naming with the SINAN / climate transforms so merges do not invent a
third geography or calendar vocabulary.


In [3]:
renamed = rename_keys(raw)
renamed.collect()


ew,state_abbrev,topic,value
str,str,str,f64
"""2020-01-26""","""AC""","""dengue""",12.0
"""2020-01-26""","""SP""","""dengue""",45.0
"""2020-01-26""","""BR""","""dengue""",30.0
"""2020-02-02""","""AC""","""dengue""",15.0


## Step 3 — Parse `ew` as Date

Strict enough for ISO `YYYY-MM-DD`; invalid strings become null (not a crash).


In [4]:
parsed = parse_ew(renamed)
parsed.collect_schema()


Schema([('ew', Date),
        ('state_abbrev', String),
        ('topic', String),
        ('value', Float64)])

## Step 4 — Floor to Sunday (shared epiweek helper)

Production GT weeks are already Sundays; `floor_ew` is a safety net so a future
extractor change cannot silently shift the calendar relative to SINAN/climate.


In [5]:
floored = floor_ew(parsed).collect()
assert floored.filter(pl.col(EW).dt.weekday() != 7).is_empty()
floored.select(EW, STATE_ABBREV, TOPIC, VALUE)


ew,state_abbrev,topic,value
date,str,str,f64
2020-01-26,"""AC""","""dengue""",12.0
2020-01-26,"""SP""","""dengue""",45.0
2020-01-26,"""BR""","""dengue""",30.0
2020-02-02,"""AC""","""dengue""",15.0


## Step 5 — Optional drop of national `BR`

Default (`keep_br=True`) keeps the national aggregate for country-level plots.
Pass `--drop-br` on the CLI when joining only to UF-level series.


In [6]:
kept = drop_national(floored.lazy(), keep_br=True).collect()
dropped = drop_national(floored.lazy(), keep_br=False).collect()
{
    "with_BR": sorted(kept[STATE_ABBREV].unique().to_list()),
    "without_BR": sorted(dropped[STATE_ABBREV].unique().to_list()),
}


{'with_BR': ['AC', 'BR', 'SP'], 'without_BR': ['AC', 'SP']}

## Step 6 — Order columns, sort — done

`build()` chains all steps. Output schema is fixed for integrity / R consumers.


In [7]:
series = build(SAMPLE, keep_br=True).collect()
assert tuple(series.columns) == OUTPUT_COLUMNS
series


ew,state_abbrev,topic,value
date,str,str,f64
2020-01-26,"""AC""","""dengue""",12.0
2020-02-02,"""AC""","""dengue""",15.0
2020-01-26,"""BR""","""dengue""",30.0
2020-01-26,"""SP""","""dengue""",45.0


## Quick look — fixture series by location

In [8]:
(
    series.group_by(STATE_ABBREV)
    .agg(pl.len().alias("n"), pl.col(VALUE).mean().alias("mean_value"))
    .sort(STATE_ABBREV)
)


state_abbrev,n,mean_value
str,u32,f64
"""AC""",2,13.5
"""BR""",1,30.0
"""SP""",1,45.0


## Optional — peek at the full local series

If you already ran `uv run arboili transform gtrends`, summarise the published
artefact (should be 27 UFs + BR).


In [9]:
full_path = PROJECT_ROOT / "data" / "google_trends" / "GoogleTrends_search_EW.parquet"
if full_path.exists():
    full = pl.read_parquet(full_path)
    print(f"rows={full.height:,}  ew={full[EW].min()} → {full[EW].max()}")
    print("locations:", full[STATE_ABBREV].n_unique())
    print("topics:", full[TOPIC].n_unique())
    print(full.head(5))
else:
    print("Full series not built yet — run: uv run arboili transform gtrends")


rows=216,720  ew=2020-01-26 → 2024-12-29
locations: 28
topics: 30
shape: (5, 4)
┌────────────┬──────────────┬──────────────────────────────┬───────┐
│ ew         ┆ state_abbrev ┆ topic                        ┆ value │
│ ---        ┆ ---          ┆ ---                          ┆ ---   │
│ date       ┆ str          ┆ str                          ┆ f64   │
╞════════════╪══════════════╪══════════════════════════════╪═══════╡
│ 2020-01-26 ┆ AC           ┆ Alteration of smell          ┆ 0.0   │
│ 2020-01-26 ┆ AC           ┆ Alteration of taste          ┆ 0.0   │
│ 2020-01-26 ┆ AC           ┆ Alteration of taste or smell ┆ 0.0   │
│ 2020-01-26 ┆ AC           ┆ Arthralgia                   ┆ 0.0   │
│ 2020-01-26 ┆ AC           ┆ Arthritis                    ┆ 0.0   │
└────────────┴──────────────┴──────────────────────────────┴───────┘


## Integrity checks

Same checks the CLI and test suite run. Expect `states` to **fail** on the
fixture (only AC, SP, BR). On the real transform output, all 27 UFs (+ BR) pass.


In [10]:
from pipeline.google_trends.integrity import run_all

for check in run_all(series, FU_PATH, keep_br=True):
    print(check)


[PASS] schema: 4 columns in published order
[PASS] unique_keys: 4 rows, all keys distinct
[PASS] weeks_start_on_sunday: all ew dates are Sundays
[FAIL] states: never appear in the series: ['AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'TO']
[PASS] value_domain: value in [0, 100] or null


## Running it for real

This notebook mirrors `transform()` in `src/pipeline/google_trends/transform.py`:

```bash
uv run arboili transform gtrends
# alias:
uv run arboili transform gt-search
```

Writes `data/google_trends/GoogleTrends_search_EW.parquet` + `.csv.gz`.
Methods: `src/pipeline/google_trends/README.md`.
